# I. Setup

The generation logic lives in `src/datasets/metaqa/`; this notebook only drives it, so
variants stay reproducible from the command line:

```
python -m src.datasets.metaqa.dataset_generator \
    --answer-len 5 --interv 2 --strategies gold retrieved --output-dir data/metaqa-multi
```


In [1]:
"""Drive `src.datasets.metaqa.dataset_generator` to build MetaQA variants.

The generator emits `{output_dir}/{baseline,meta-icl}/standard/{train,val,test}.json`
in the GraphQA record schema. Two axes are exposed: the answer-set size window and
which seeding strategies (`gold`, `retrieved`, or both) appear in `algorithm`.
"""

%load_ext autoreload
%autoreload 2

import json
import os
import sys
from collections import Counter

if os.path.basename(os.getcwd()) == "exploration":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from src.datasets.metaqa.dataset_generator import main
from src.models.meta_icl import answer_list, set_scores

print("cwd:", os.getcwd())


cwd: /home/m.ortega/repositories/graph-representation-learning-for-llm


In [6]:
# Questions are kept when their answer count falls in
# [ANSWER_LEN - INTERV_ANSWER_LEN, ANSWER_LEN + INTERV_ANSWER_LEN], clipped at 1.
ANSWER_LEN = 1
INTERV_ANSWER_LEN = 0

# Subset of ["gold", "retrieved"]; each value emitted becomes an `algorithm` value.
# STRATEGIES = ["gold", "retrieved"]
STRATEGIES = ["gold"]

OUTPUT_DIR = "data/metaqa-gold"
SEED = 42


# II. Creating Variants

In [7]:
main(
    strategies=STRATEGIES,
    answer_len=ANSWER_LEN,
    interv_answer_len=INTERV_ANSWER_LEN,
    output_dir=OUTPUT_DIR,
    seed=SEED,
)


Loading KB from data/metaqa/kb.txt ...


subgraphs [train]: 100%|██████████| 3999/3999 [00:01<00:00, 2012.29it/s]


  subgraphs [train]: kept 3999, dropped 0 | gold answers-in-graph 100.0%


subgraphs [val]: 100%|██████████| 399/399 [00:00<00:00, 1953.53it/s]


  subgraphs [val]: kept 399, dropped 0 | gold answers-in-graph 100.0%


subgraphs [test]: 100%|██████████| 600/600 [00:00<00:00, 1999.77it/s]


  subgraphs [test]: kept 600, dropped 0 | gold answers-in-graph 100.0%
baseline/train: 3999 rows -> data/metaqa-gold/baseline/standard/train.json
baseline/val: 399 rows -> data/metaqa-gold/baseline/standard/val.json
baseline/test: 600 rows -> data/metaqa-gold/baseline/standard/test.json
meta-icl/train: 3999 rows -> data/metaqa-gold/meta-icl/standard/train.json
meta-icl/val: 399 rows -> data/metaqa-gold/meta-icl/standard/val.json
meta-icl/test: 600 rows -> data/metaqa-gold/meta-icl/standard/test.json


In [4]:
rows = json.load(open(f"{OUTPUT_DIR}/baseline/standard/train.json"))

for row in rows:
    # `make_collator` finds the label span by character offset from the end.
    assert row["question"].endswith("A: "), row["id"]
    answers = answer_list(row["answer"])
    assert len(answers) == int(row["nanswers"]), row["id"]
    if row["algorithm"] == "gold":
        assert all(a in row["question"] for a in row["answer"].rstrip(".").split(" | ")), row["id"]

print(f"{len(rows)} rows verified")
print("algorithms: ", Counter(r["algorithm"] for r in rows))
print("tasks:      ", Counter(r["task"] for r in rows))
print("nanswers:   ", sorted(Counter(int(r["nanswers"]) for r in rows).items()))
print("answer chars: max =", max(len(r["answer"]) for r in rows))


7983 rows verified
algorithms:  Counter({'gold': 3999, 'retrieved': 3984})
tasks:       Counter({'2hop': 2662, '3hop': 2661, '1hop': 2660})
nanswers:    [(3, 3239), (4, 2108), (5, 1112), (6, 898), (7, 626)]
answer chars: max = 227


In [5]:
# Whole-string exact match is all-or-nothing on these targets; val/set_f1 and
# val/hits_at_1 are what MetaICL logs for partial credit.
label = rows[0]["answer"]
partial = " | ".join(label.rstrip(".").split(" | ")[:-1]) + "."

print("label:  ", label)
print("partial:", partial)
for name, pred in [("perfect", label), ("partial", partial), ("empty", ".")]:
    precision, recall, f1, hits1 = set_scores(pred, label)
    print(f"{name:8s} precision={precision:.2f} recall={recall:.2f} f1={f1:.2f} hits@1={hits1:.2f}")


label:   Don't Look Now | Frenchman's Creek | Mayerling | My Cousin Rachel | Rebecca | The Birds | The Scapegoat.
partial: Don't Look Now | Frenchman's Creek | Mayerling | My Cousin Rachel | Rebecca | The Birds.
perfect  precision=1.00 recall=1.00 f1=1.00 hits@1=1.00
partial  precision=1.00 recall=0.86 f1=0.92 hits@1=1.00
empty    precision=0.00 recall=0.00 f1=0.00 hits@1=0.00
